#**Generative Music with a Transformer (MIDI Generation)**




In this notebook we will train a decoder-only Transformer to generate piano music. The approach is almost identical to what we did in the Mini GPT notebook — we tokenize a sequence, train a Transformer to predict the next token, and then generate new sequences autoregressively. The only difference is that instead of text tokens, we work with **MIDI tokens** that represent musical events (which note to play, how hard to press it, when to release it, and how long to wait between events).

This is based on the *Music Transformer* paper (Huang et al., 2018) and the corresponding [Keras tutorial](https://keras.io/examples/generative/midi_generation_with_transformer/).

# *Install Dependencies and Import Libraries*

In [ ]:
!pip install -qq midi_neural_processor
!pip install -qq keras_hub
!pip install -qq "keras>=3.6.0"

# For audio playback
!sudo apt-get -qq install -y fluidsynth 2> /dev/null
!pip install -qq pyfluidsynth scipy pretty_midi

In [ ]:
import os
import random
import tempfile

import keras
import midi_neural_processor.processor as midi_tokenizer
import numpy as np
from keras import callbacks, layers, ops, optimizers, utils
from keras_hub import layers as hub_layers
from os import path

# *Understanding MIDI Tokenization*

MIDI (Musical Instrument Digital Interface) files don't store audio waveforms — they store **events** that describe what a musician does. Think of it like sheet music in digital form. The `midi_neural_processor` library converts these events into integer tokens, just like we converted characters or words into integers in our text models.

There are four types of MIDI events, each mapped to a range of integer values:

| Event Type | What It Means | Example |
|-----------|--------------|--------|
| **Note On** (0–127) | Start playing a note (128 possible pitches, from very low to very high) | Token 60 = start playing Middle C |
| **Note Off** (128–255) | Stop playing a note | Token 188 = stop playing Middle C |
| **Time Shift** (256–355) | Wait before the next event (in 10ms increments, up to 1 second) | Token 306 = wait 500ms |
| **Velocity** (356–387) | How hard the next note is struck (32 levels of loudness) | Token 370 = medium-loud |

A short melody like playing Middle C for half a second, then E for half a second, might look like:

```
Token 370  → Set velocity (medium)
Token 60   → Note On: Middle C
Token 306  → Wait 500ms
Token 188  → Note Off: Middle C
Token 64   → Note On: E
Token 306  → Wait 500ms
Token 192  → Note Off: E
```

This is a **discrete sequence of integers** — exactly what a Transformer is designed to model. The same next-token prediction approach that generates text can generate music.

# *Configuration*

We set up a configuration object with our model hyperparameters and special tokens (padding, start-of-sequence, end-of-sequence) — similar to the `[start]` and `[end]` tokens we used in our machine translation notebook.

In [ ]:
# Total count of possible MIDI event tokens
event_range = midi_tokenizer.RANGE_NOTE_ON
event_range += midi_tokenizer.RANGE_NOTE_OFF
event_range += midi_tokenizer.RANGE_TIME_SHIFT
event_range += midi_tokenizer.RANGE_VEL

# We add 3 special tokens on top of the MIDI events: padding, start-of-sequence, end-of-sequence
CONFIG = utils.Config(
    max_sequence_len=2048,
    embedding_dim=256,
    num_transformer_blocks=6,
    batch_size=6,
    token_pad=event_range,
    token_start_of_sentence=event_range + 1,
    token_end_of_sentence=event_range + 2,
    vocabulary_size=event_range + 3,
    model_out="tmp/music_transformer.keras",
    seed=42,
)
utils.set_random_seed(CONFIG.seed)

print(f'We have {event_range} unique MIDI event tokens, plus 3 special tokens = {CONFIG.vocabulary_size} total vocabulary size.')
print(f'Maximum sequence length: {CONFIG.max_sequence_len} tokens')

# *Download and Preprocess the Maestro Dataset*

We will use the **Maestro dataset** (MIDI and Audio Edited for Synchronous TRacks and Organization) — a collection of ~1,200 classical piano performances from international piano competitions. The download is ~200 MB of MIDI files.

In [ ]:
def download_maestro(output_dir=None):
    """Download the Maestro MIDI dataset."""
    output_dir = tempfile.mkdtemp() if output_dir is None else output_dir
    os.makedirs(output_dir, exist_ok=True)

    dir = utils.get_file(
        origin="https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip",
        extract=True,
    )

    midi_files, file_paths = set(), list()
    for root, _, files in os.walk(dir):
        for file in files:
            if file.lower().endswith(".midi") or file.lower().endswith(".mid"):
                midi_files.add(path.join(root, file))

    for file in sorted(midi_files):
        file_paths.append(new_path := path.join(output_dir, path.basename(file)))
        os.rename(file, new_path)
    return file_paths

paths = list(sorted(download_maestro(output_dir="datasets/maestro")))
output_dir = path.dirname(paths[0])

print(f'Downloaded {len(paths)} MIDI files.')

Split into training and validation sets (90/10 split).

In [ ]:
indices = np.random.permutation(len(paths))
split = int(len(paths) * 0.1)
train_paths = [paths[i] for i in indices[split:]]
val_paths = [paths[i] for i in indices[:split]]

print(f'Training files: {len(train_paths)}, Validation files: {len(val_paths)}')

# *Listen to a Training Sample*

Before we do any modeling, let's hear what the raw training data sounds like. We convert a MIDI file to audio using FluidSynth (a software synthesizer).

In [ ]:
import pretty_midi
from scipy.io.wavfile import write as write_wav
from IPython.display import Audio

def visualize_midi(midi_path, sampling_rate=16000, seconds=15, out_dir=None):
    """Convert a MIDI file to audio for playback."""
    pretty_midi_file = pretty_midi.PrettyMIDI(midi_path)
    waveform = pretty_midi_file.fluidsynth(fs=sampling_rate)[: seconds * sampling_rate]

    if out_dir is None:
        return Audio(waveform, rate=sampling_rate)

    os.makedirs(out_dir, exist_ok=True)
    audio_path = path.join(out_dir, path.basename(midi_path).split(".")[0] + ".wav")
    write_wav(audio_path, sampling_rate, (waveform * 32767).astype(np.int16))
    return audio_path

# Listen to the first 15 seconds of a training sample
visualize_midi(train_paths[0])

# *Tokenize and Build the Dataset*

We now tokenize every MIDI file into integer sequences using `midi_neural_processor`. This uses multiprocessing to speed things up. The result is a list of numpy arrays — one per MIDI file — containing the token sequences.

This step may take a few minutes on the first run, but the results are cached to disk.

In [ ]:
def encode_midi_task(midi_path):
    """Tokenize a single MIDI file (used by multiprocessing pool)."""
    import midi_neural_processor.processor as midi_tokenizer
    return midi_tokenizer.encode_midi(midi_path)

def preprocess_midi_files(file_paths, save_dir=None):
    """Preprocess a list of MIDI files and cache the results."""
    from multiprocessing import Pool, cpu_count

    save_dir = path.dirname(file_paths[0]) if save_dir is None else save_dir
    os.makedirs(save_dir, exist_ok=True)

    # Check if already preprocessed
    output_file = path.join(save_dir, "notes.npz")
    if path.exists(output_file):
        npz_file = np.load(output_file)
        return [npz_file[key] for key in npz_file.keys()]

    # Preprocess in parallel
    progbar = utils.Progbar(len(file_paths), unit_name="MIDI_file", interval=5)
    pool = Pool(cpu_count() - 1)
    all_notes = []
    for notes in pool.imap_unordered(encode_midi_task, file_paths):
        progbar.add(1)
        all_notes.append(np.array(notes))

    np.savez(output_file, *all_notes)
    return all_notes

train_midis = preprocess_midi_files(train_paths, path.join(output_dir, "train"))
val_midis = preprocess_midi_files(val_paths, path.join(output_dir, "val"))

print(f'Tokenized {len(train_midis)} training files and {len(val_midis)} validation files.')
print(f'Example: first training file has {len(train_midis[0])} tokens.')

Now we wrap the tokenized data in a Dataset class. Just like in our Shakespeare and Mini GPT notebooks, each training example is a pair of sequences offset by one token: the model sees tokens 0 through N-1 as input and predicts tokens 1 through N as output.

In [ ]:
class MidiDataset(utils.PyDataset):
    """A dataset that yields batches of (input_sequence, target_sequence) from tokenized MIDI."""

    def __init__(self, encoded_midis, batch_size=CONFIG.batch_size, max_sequence_len=CONFIG.max_sequence_len):
        super(MidiDataset, self).__init__()
        self.batch_size = batch_size
        self.max_sequence_len = max_sequence_len
        self.encoded_midis = encoded_midis
        batches, last_batch_size = divmod(len(encoded_midis), batch_size)
        self._num_batches = batches + int(last_batch_size > 0)

    def __len__(self):
        return self._num_batches

    def __getitem__(self, idx):
        batch = random.sample(self.encoded_midis, k=self.batch_size)
        batch_data = [self._get_sequence(midi, self.max_sequence_len + 1) for midi in batch]
        batch_data = np.array(batch_data)

        # Input is tokens 0..N-1, target is tokens 1..N (offset by 1)
        return batch_data[:, :-1], batch_data[:, 1:]

    def _get_sequence(self, data, max_length):
        """Extract a random subsequence from a MIDI file and pad if needed."""
        if len(data) > max_length:
            start = random.randrange(0, len(data) - max_length)
            data = data[start : start + max_length]
        elif len(data) < max_length:
            data = np.append(data, CONFIG.token_end_of_sentence)

        if len(data) < max_length:
            data = np.concatenate((data, np.full(max_length - len(data), CONFIG.token_pad)))
        return np.asanyarray(data, dtype="int32")

train_dataset = MidiDataset(train_midis)
val_dataset = MidiDataset(val_midis)

print(f'Training batches: {len(train_dataset)}, Validation batches: {len(val_dataset)}')

# *The Music Transformer Architecture*

Our model is a decoder-only Transformer — the same high-level structure as our Mini GPT. The key architectural difference is the use of **relative positional attention** instead of fixed positional embeddings.

### Why Relative Attention for Music?

Standard positional embeddings encode **where** a token is in the sequence (position 0, position 1, ...). Relative attention instead encodes **how far apart** two tokens are. This is more natural for music: a melody that starts at beat 1 should sound the same as the same melody starting at beat 5. The relationships between notes matter more than their absolute positions.

Concretely, when computing attention between tokens at positions `i` and `j`, standard attention adds a fixed embedding for each position. Relative attention instead adds a learned embedding for the distance `i - j`. This makes the model translation-invariant in time — a useful inductive bias for music.

In [ ]:
@keras.utils.register_keras_serializable()
class RelativeGlobalAttention(layers.Layer):
    """
    Multi-head attention with relative positional encoding.
    From Music Transformer (Huang et al., 2018): https://arxiv.org/abs/1809.04281
    """

    def __init__(self, num_heads, embedding_dim, max_sequence_len, **kwargs):
        super().__init__(**kwargs)
        self.key_length = None
        self.max_sequence_len = max_sequence_len
        self.relative_embedding = None
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim
        self.head_dim = embedding_dim // num_heads
        self.query_dense = layers.Dense(int(self.embedding_dim))
        self.key_dense = layers.Dense(int(self.embedding_dim))
        self.value_dense = layers.Dense(int(self.embedding_dim))
        self.output_dense = layers.Dense(embedding_dim, name="output")

    def build(self, input_shape):
        self.query_length = input_shape[0][1]
        self.key_length = input_shape[1][1]
        self.relative_embedding = self.add_weight(
            (self.max_sequence_len, int(self.head_dim)), name="relative_embedding"
        )

    def _apply_dense_layer_and_split_heads(self, inputs, dense_layer):
        inputs = dense_layer(inputs)
        new_shape = ops.shape(inputs)
        reshaped = ops.reshape(inputs, (new_shape[0], new_shape[1], self.num_heads, -1))
        return ops.transpose(reshaped, (0, 2, 1, 3))

    def call(self, inputs, mask=None):
        # Compute Q, K, V with shape: (batch, heads, sequence, features)
        query = self._apply_dense_layer_and_split_heads(inputs[0], self.query_dense)
        key = self._apply_dense_layer_and_split_heads(inputs[1], self.key_dense)
        value = self._apply_dense_layer_and_split_heads(inputs[2], self.value_dense)

        # Standard scaled dot-product attention scores
        attention_scores = ops.matmul(query, ops.transpose(key, [0, 1, 3, 2]))

        # Add relative positional encoding to the attention scores
        start_idx = max(0, self.max_sequence_len - ops.shape(query)[2])
        relative_embedding = self.relative_embedding[start_idx:, :]
        attention_scores += self._compute_attention_scores(query, relative_embedding)
        logits = attention_scores / ops.sqrt(self.head_dim)

        if mask is not None:
            logits += ops.cast(mask, "float32") * -1e9

        attention_weights = ops.nn.softmax(logits, axis=-1)
        attention_output = ops.matmul(attention_weights, value)

        # Merge heads back together
        merged_attention = ops.transpose(attention_output, (0, 2, 1, 3))
        merged_attention = ops.reshape(
            merged_attention, (ops.shape(merged_attention)[0], -1, self.embedding_dim)
        )
        output = self.output_dense(merged_attention)
        return output, attention_weights

    def _compute_attention_scores(self, query, relative_embedding):
        relative_scores = ops.einsum("bhld, md->bhlm", query, relative_embedding)
        relative_scores = self._apply_mask_to_relative_scores(relative_scores)
        return self._skew_attention_scores(relative_scores)

    def _apply_mask_to_relative_scores(self, scores):
        mask = ops.flip(
            ops.tri(scores.shape[-2], scores.shape[-1], dtype="float32"), axis=1
        )
        return mask * scores

    def _skew_attention_scores(self, scores):
        padded_scores = ops.pad(scores, ((0, 0), (0, 0), (0, 0), (1, 0)))
        padded_shape = ops.shape(padded_scores)
        reshaped_scores = ops.reshape(
            padded_scores, (-1, padded_shape[1], padded_shape[-1], padded_shape[-2])
        )
        skewed_scores = reshaped_scores[:, :, 1:, :]

        if self.key_length > self.query_length:
            size_diff = self.key_length - self.query_length
            return ops.pad(skewed_scores, [[0, 0], [0, 0], [0, 0], [0, size_diff]])
        else:
            return skewed_scores[:, :, :, : self.key_length]

### Decoder Layer

Each decoder layer has the same structure we saw in our Mini GPT: self-attention followed by a feed-forward network, with layer normalization and residual connections. The only difference is that the attention mechanism here uses relative positional encoding.

In [ ]:
@keras.utils.register_keras_serializable()
class DecoderLayer(layers.Layer):
    def __init__(self, embedding_dim, num_heads, max_sequence_len, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.max_sequence_len = max_sequence_len

        self.relative_global_attention_1 = RelativeGlobalAttention(
            num_heads, embedding_dim, max_sequence_len
        )

        self.feed_forward_network_pre = layers.Dense(self.embedding_dim // 2, "relu")
        self.feed_forward_network_pos = layers.Dense(self.embedding_dim)

        self.layer_normalization_1 = layers.LayerNormalization(epsilon=1e-6)
        self.layer_normalization_2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout_1 = layers.Dropout(dropout)
        self.dropout_2 = layers.Dropout(dropout)

    def call(self, inputs, mask=None, training=False):
        attention_out, attention_weights = self.relative_global_attention_1(
            (inputs, inputs, inputs), mask=mask
        )
        attention_out = self.dropout_1(attention_out, training=training)
        attention_out_normalized = self.layer_normalization_1(attention_out + inputs)

        ffn_out = self.feed_forward_network_pre(attention_out_normalized)
        ffn_out = self.feed_forward_network_pos(ffn_out)
        ffn_out = self.dropout_2(ffn_out, training=training)
        out = self.layer_normalization_2(attention_out_normalized + ffn_out)

        return out, attention_weights

### Full Decoder Stack

We stack 6 decoder layers, preceded by a token embedding layer and sine-based positional encoding. This is the same stacking pattern we used in Mini GPT (where we had 8 layers).

In [ ]:
@keras.utils.register_keras_serializable()
class Decoder(layers.Layer):
    def __init__(self, embedding_dim, vocabulary_size, max_sequence_len, num_blocks, dropout):
        super(Decoder, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_blocks = num_blocks

        self.embedding = layers.Embedding(vocabulary_size, self.embedding_dim)
        self.positional_encoding = hub_layers.SinePositionEncoding()

        self.decode_layers = [
            DecoderLayer(
                embedding_dim, embedding_dim // 64, max_sequence_len, dropout=dropout
            )
            for _ in range(num_blocks)
        ]
        self.dropout = layers.Dropout(dropout)

    def call(self, inputs, mask=None, training=False, return_attention_weights=False):
        weights = []

        # Token embedding + positional encoding
        x = self.embedding(inputs)
        x = x * ops.sqrt(ops.cast(self.embedding_dim, "float32"))
        x = x + self.positional_encoding(x)
        x = self.dropout(x, training=training)

        # Pass through all transformer blocks
        for i in range(self.num_blocks):
            x, w = self.decode_layers[i](x, mask=mask, training=training)
            weights.append(w)

        if return_attention_weights:
            return x, weights
        return x

### The Music Transformer Model

This wraps everything together: the decoder stack, an output Dense layer, causal masking (the same lower-triangular mask we saw in Mini GPT — each position can only attend to earlier positions), and a `generate()` method for inference.

The `generate()` method uses **top-k sampling**: instead of always picking the most likely next token (greedy decoding, which produced the repetitive output we saw in Mini GPT), we sample from the top *k* most likely tokens. This introduces controlled randomness that makes the output more creative and varied.

In [ ]:
@keras.utils.register_keras_serializable()
class MusicTransformerDecoder(keras.Model):
    def __init__(
        self,
        embedding_dim=CONFIG.embedding_dim,
        vocabulary_size=CONFIG.vocabulary_size,
        num_blocks=CONFIG.num_transformer_blocks,
        max_sequence_len=CONFIG.max_sequence_len,
        dropout=0.2,
    ):
        super(MusicTransformerDecoder, self).__init__()
        self.embedding_dim = embedding_dim
        self.vocabulary_size = vocabulary_size
        self.num_blocks = num_blocks
        self.max_sequence_len = max_sequence_len

        self.decoder = Decoder(
            embedding_dim, vocabulary_size, max_sequence_len, num_blocks, dropout
        )
        self.fc = layers.Dense(self.vocabulary_size, activation=None, name="output")

    @staticmethod
    def get_look_ahead_mask(max_sequence_len, inputs):
        """Create the causal mask — each position can only attend to earlier positions."""
        sequence_length = min(max_sequence_len, inputs.shape[1])
        sequence_mask = ops.logical_not(
            ops.tri(sequence_length, sequence_length, dtype="bool")
        )
        inputs = ops.cast(inputs[:, None, None, :], "int32")
        output_pad_tensor = ops.ones_like(inputs) * CONFIG.token_pad
        decoder_output_mask = ops.equal(inputs, output_pad_tensor)
        return ops.cast(ops.logical_or(decoder_output_mask, sequence_mask), "int32")

    def call(self, inputs, training=False):
        mask = self.get_look_ahead_mask(self.max_sequence_len, inputs)
        decoding = self.decoder(
            inputs, mask=mask, training=training, return_attention_weights=False
        )
        return self.fc(decoding)

    def generate(self, inputs: list, length=CONFIG.max_sequence_len, top_k=5):
        """Generate a sequence of MIDI tokens using top-k sampling."""
        inputs = ops.convert_to_tensor([inputs])

        def generate_token(inputs, end_idx):
            distribution = ops.stop_gradient(self.call(inputs)[0, end_idx])
            top_k_distribution, top_k_indices = ops.top_k(distribution, k=top_k)
            new_token_idx = keras.random.categorical(top_k_distribution[None, :], 1)
            return ops.take(top_k_indices, new_token_idx[0])

        added_tokens = min(length, self.max_sequence_len - inputs.shape[1])
        progbar = utils.Progbar(added_tokens, unit_name="token", interval=5)

        out = ops.pad(inputs, ((0, 0), (0, added_tokens)), "constant", CONFIG.token_pad)

        for token_idx in range(inputs.shape[1] - 1, inputs.shape[1] - 1 + added_tokens):
            token = ops.cast(generate_token(out, end_idx=token_idx), out.dtype)
            out = ops.scatter_update(out, ((0, token_idx + 1),), token)
            progbar.add(1)

        return ops.convert_to_numpy(out[0])

    def get_config(self):
        atts = ["embedding_dim", "vocabulary_size", "num_blocks", "max_sequence_len"]
        return {a: getattr(self, a) for a in atts}

    @classmethod
    def from_config(cls, config):
        return cls(**config)

# *Training*

We use the same learning rate warmup strategy from our Mini GPT notebook — start with a very small learning rate and gradually increase it over the first 4,000 steps to avoid exploding gradients in the early stages of training.

Training 15 epochs on a T4 GPU takes roughly 10-15 minutes.

In [ ]:
@keras.utils.register_keras_serializable()
def train_loss(y_true, y_pred):
    """Masked cross-entropy loss — ignores padding tokens."""
    mask = ops.cast(ops.logical_not(ops.equal(y_true, CONFIG.token_pad)), "float32")
    y_true = ops.one_hot(ops.cast(y_true, "int32"), CONFIG.vocabulary_size)
    return ops.categorical_crossentropy(y_true, y_pred, from_logits=True) * mask

@keras.utils.register_keras_serializable()
class CustomSchedule(optimizers.schedules.LearningRateSchedule):
    """Learning rate warmup schedule (same idea as Mini GPT)."""
    def __init__(self, embedding_dim, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.embedding_dim = embedding_dim
        self.warmup_steps = warmup_steps
        self._embedding_dim = ops.cast(self.embedding_dim, "float32")
        self._lr_adjust = 0.1 if keras.backend.backend() == "torch" else 1.0

    def get_config(self):
        return {"embedding_dim": self.embedding_dim, "warmup_steps": self.warmup_steps}

    def __call__(self, step):
        step_rsqrt = ops.rsqrt(ops.cast(step, "float32"))
        warmup_adjust = step * (self.warmup_steps**-1.5)
        output = ops.rsqrt(self._embedding_dim) * ops.minimum(step_rsqrt, warmup_adjust)
        return self._lr_adjust * output

In [ ]:
def train_model(model, train_ds, val_ds, epochs=15):
    learning_rate = CustomSchedule(CONFIG.embedding_dim)
    optimizer = optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

    model.compile(optimizer=optimizer, loss=train_loss)

    save_cb = callbacks.ModelCheckpoint(CONFIG.model_out, save_best_only=True)
    model.fit(
        train_ds, validation_data=val_ds, epochs=epochs, callbacks=[save_cb], verbose=2
    )
    return model

# Train (or load a previously saved model)
if path.exists(CONFIG.model_out):
    model = keras.models.load_model(CONFIG.model_out)
else:
    model = train_model(MusicTransformerDecoder(), train_dataset, val_dataset)

# *Generate Music!*

We seed the model with 25 tokens from a validation MIDI file and let it generate up to 1,024 new tokens using top-k sampling (k=15). The generated token sequence is then decoded back into a MIDI file that we can listen to.

In [ ]:
def generate_music(model, seed_path, length=1024, out_dir=None, top_k=None):
    """Generate music from a seed MIDI file."""
    out_dir = out_dir if out_dir is not None else tempfile.mkdtemp()
    os.makedirs(out_dir, exist_ok=True)

    # Take 25 tokens from the middle of the seed file as our starting prompt
    inputs = midi_tokenizer.encode_midi(seed_path)[100:125]
    print(f"Seed tokens: {inputs}")

    # Generate!
    result = model.generate(inputs, length=length, top_k=top_k)

    output_path = path.join(out_dir, path.basename(seed_path).split(".")[0] + ".mid")
    midi_tokenizer.decode_midi(result, output_path)
    return output_path

output_file = generate_music(model, val_paths[-1], out_dir="tmp/", top_k=15)
print(f"Generated MIDI saved to: {output_file}")

Listen to the generated music...

In [ ]:
visualize_midi(output_file)

Let's also try with a different seed and a smaller top-k (less randomness)...

In [ ]:
output_file_2 = generate_music(model, val_paths[0], out_dir="tmp/", top_k=5)
visualize_midi(output_file_2)

# *Discussion*

### What We Built
A decoder-only Transformer that generates piano music token by token — the same autoregressive approach we used for text generation, but applied to MIDI events instead of words or characters.

### Quality Considerations
The generated music should have recognizable musical structure (chords, melodic phrases, rhythmic patterns) but won't sound like a polished composition. With more training data, more epochs, and a larger model, quality improves significantly. State-of-the-art music generation models (like Google's MusicLM or Meta's MusicGen) use much larger architectures and datasets, but the core principle is the same.

### Key Takeaways
- **Tokenization is the bridge**: just as we tokenized text into integers for language models, we tokenized musical events into integers here. The Transformer doesn't "know" it's generating music — it's just predicting the next integer in a sequence.
- **Relative attention** is a useful inductive bias for domains where relative position matters more than absolute position (music, but also potentially code, speech, etc.).
- **Top-k sampling** produces much more interesting output than greedy decoding, at the cost of some coherence. The choice of k controls this trade-off.